Unifilar LV Diagram - Riodades
=
This notebook draws the low-voltage single-line diagram (unifilar BT) of UFV Riodades (4 UFVs, 160 inverters, 2858 strings) from Excel tables. It computes where every inverter block, inverter tag, AC cable tag, string tag and module power label goes, and writes AutoCAD scripts (`.scr`) plus an Excel copy of the string table. It doesn't read any DWG.

It is a copy of the Sendim notebook with these differences:
- String count (`Qty. of Strings`), pattern (`PD`) and module power (`POT`) come per string from the Riodades `STR` sheet, so there are no hardcoded PST / inverter overrides.
- Only patterns 1 (18 strings) and 2 (17 strings).
- String distances come from the same `STR` sheet.
- The layout parameters (`Parametros`) are still read from the **Sendim** workbook.

**Workflow overview:**
1. Read the parameters (Sendim workbook) and the `INV` / `STR` sheets (Riodades workbook), compute the inverter, inverter tag and AC cable tag positions and write the three inverter scripts.
2. Read the string distances.
3. Build the string table, apply the T-C-B exception and order the strings inside each inverter by type and distance.
4. Put each string on a slot (row) of its inverter using `PATTERNS`, then write the string tag and module power scripts and the Excel table.

**Input prerequisites:**
- `UFV Sendim - Unifilar BT.xlsx` (Sendim project folder):
  - `Parametros` sheet: columns C:D, header on row 2. The values are read **by row position** from column D (`df_parameter.iloc[row, 1]`), so the row order must not change. What each row is used for (names inferred from the code):
    - 0 X origin, 1 Y origin, 2 X pitch per switchboard (PCS/QG pair), 3 X pitch between DJ columns, 4 Y pitch between DJs, 5 DJs per column
    - 6 / 7 inverter tag X / Y offset from the inverter
    - 8 not used
    - 9 / 10 string tag X / Y offset from the inverter, 11 string tag Y pitch per slot
    - 12 / 13 module power X / Y offset from the inverter, 14 module power Y pitch per slot
    - 15 AC cable tag X offset, 16 / 17 AC cable tag Y offset for QG 1 / QG 2, 18 AC cable tag Y step per DJ
- `UFV Riodades - Unifilar BT.xlsx` (Riodades project folder):
  - `INV` sheet: columns A:K, header on row 2. The **first column** is the inverter tag (`TAG`, e.g. `INV-1.1.01`), plus `UFV`, `PCS`, `QG`, `DJ`, `INV` and `Dist` (AC cable length, m).
  - `STR` sheet: columns A:M, header on row 1, one row per string: `Tag completa`, `UFV`, `PST`, `INV`, `STR`, `Type` (T / B / C), `Type N` (1 / 2 / 3), `POT`, `Qty. of Strings` and `PD` (1 or 2 only). Column H must be the string tag and column J the distance, because the distance is read again by column letter.
- String tags must be fixed-width `STR-U.P.II.SS` (1-digit UFV and PST, 2-digit INV, e.g. `STR-1.1.01.05`). The inverter tag is cut from characters 4-10 (`INV-1.1.01`) and the PST tag from characters 4-7 (`PST-1.1`). A 2-digit UFV or PST breaks both.

**Target drawing prerequisites** (to run the generated scripts, which use no DWG as input):
- A block named `Inversor` is defined in the drawing and is **not** set to *Scale uniformly*: the insert line answers X scale, Y scale and rotation (`1 1 0`). If the block has attributes, set `ATTREQ = 0`, because the script gives no attribute values.
- The layers `EMF_unifilar_inversor`, `EMF_TAG_inversor`, `CA_Cabos`, `TAG_strings` and `Potência_Módulo` exist. Each script starts with `-LAYER SET <layer>`, which fails on a missing layer.
- MText is written with the current text style (only height, justification, rotation and width are set).

Inputs, inverter positions and inverter scripts
=
- `inv_step = 9842.2311`: vertical distance between two UFV blocks. UFVs are stacked with `inv_step · (8 − UFV)`. The `8` is Sendim's UFV count; with Riodades' 4 UFVs the whole diagram just sits 4 steps higher (UFV 4 at 4 steps, not 0).
- `PATTERNS`: slot rows (1-based) for 18 (pattern 1) and 17 (pattern 2) strings. The gaps separate the MPPT groups.
- `qty_strings` and `math` are not used.
- Warning: `Parametros` is read from the **Sendim** workbook. It only works if Riodades uses the same diagram template; otherwise point it to the Riodades workbook.
- Inverter block position:
  - `X = X0 + (2·PCS − 3 + QG) · pitch_switchboard + pitch_DJcolumn · trunc((DJ − 1) / DJs_per_column)`. Each PCS/QG pair gets its own column band (PCS 1/QG 1 → 0, PCS 1/QG 2 → 1, PCS 2/QG 1 → 2, ...), and the DJs wrap into a new column every `DJs_per_column`.
  - `Y = inv_step · (8 − UFV) + Y0 + ((DJ − 1) mod DJs_per_column) · pitch_DJ`. The expression `DJ − ceil(DJ/n)·n + n − 1` is just `(DJ − 1) mod n`.
- `X tag` / `Y tag`: inverter tag position = inverter position + fixed offset (rows 6 / 7).
- `X cabo`: AC cable tag column, one per PCS/QG pair (row 15 offset).
- The table is sorted by `UFV, PCS, QG, DJ` (the order of the script lines).
- `Y cabo` (AC cable tag Y), row by row:
  - DJ 1 of QG 1 → `inv_step·(8 − UFV) + Y0 + offset_QG1` (row 16); DJ 1 of QG 2 → same with row 17.
  - Other DJs → the DJ 1 value of their QG + `(DJ − 1) · step` (row 18), so the cable tags form one column per QG.
  - `test` is assigned in the loop but never used.
- `TAG Cabo CA` = `CLV-UFV.PCS.INV.(L1-L2-L3) - <Dist>m` (INV zero-padded to 2 digits, length rounded to 0.1 m).
- `Script 1`: `-insert Inversor X,Y 1 1 0` (X scale 1, Y scale 1, rotation 0).
- `Script 2`: MText with the inverter tag (`TAG`), height 30, justification ML, rotation −90.
- `Script 3`: MText with the AC cable tag, height 20, justification TL, rotation 0.
- Writes `Riodades - Script Inversor.scr` (layer `EMF_unifilar_inversor`), `Riodades - Script TAG Inversor.scr` (layer `EMF_TAG_inversor`) and `Riodades - Script Cabos CA.scr` (layer `CA_Cabos`) in `Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar`.

In [1]:
import time

import pandas as pd

import math

import numpy as np

start = time.time()

qty_strings = 18

inv_step = 9842.2311

PATTERNS = {
    1: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25,26],
    2: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25]
}

df_parameter = pd.read_excel(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\302 - Zagope - UFV Sendim\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Sendim - Unifilar BT.xlsx", sheet_name="Parametros", usecols="C:D", header=1)

with pd.ExcelFile(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\309 - Zagope - UVF Riodades\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Riodades - Unifilar BT.xlsx") as xlsx:
    df_inv = pd.read_excel(xlsx, sheet_name="INV", usecols="A:K", header=1)
    df_str = pd.read_excel(xlsx, sheet_name="STR", usecols="A:M")


df_inv['X'] = df_parameter.iloc[0, 1]+(df_inv["PCS"]*2-3+df_inv["QG"])*df_parameter.iloc[2, 1]+df_parameter.iloc[3, 1]*np.trunc((df_inv["DJ"]-1)/df_parameter.iloc[5, 1])
df_inv['Y'] = (inv_step * (8 - df_inv['UFV'])) + (df_parameter.iloc[1, 1] + (df_inv['DJ'] - np.ceil(df_inv['DJ'] / df_parameter.iloc[5, 1]) * df_parameter.iloc[5, 1] + df_parameter.iloc[5, 1] - 1.0) * df_parameter.iloc[4, 1])
df_inv['X tag'] = df_inv['X'] + df_parameter.iloc[6, 1]
df_inv['Y tag'] = df_inv['Y'] + df_parameter.iloc[7, 1]
df_inv['X cabo'] = df_parameter.iloc[0, 1]+(df_inv['PCS']*2+df_inv['QG']-3)*df_parameter.iloc[2, 1]+df_parameter.iloc[15, 1]
df_inv['Y cabo'] = 0.0

df_inv = df_inv.sort_values(by=["UFV", "PCS", "QG", "DJ"]).reset_index(drop=True)

for i in range(len(df_inv)):
    if (df_inv.loc[i, 'QG'] == 1) and (df_inv.loc[i, 'DJ'] == 1):
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[16, 1])
    elif (df_inv.loc[i, 'QG'] == 2) and (df_inv.loc[i, 'DJ'] == 1):
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[17, 1])
    elif df_inv.loc[i, 'QG'] == 1:
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[16, 1]) + (df_inv.loc[i, 'DJ']-1)*df_parameter.iloc[18, 1]
        test = df_inv.loc[i, 'Y cabo']
    else:
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[17, 1]) + (df_inv.loc[i, 'DJ']-1)*df_parameter.iloc[18, 1]


df_inv['TAG Cabo CA'] = "CLV-" + df_inv["UFV"].astype(str) + "." + \
    df_inv["PCS"].astype(str) + "." + \
    df_inv['INV'].astype(str).str.zfill(2) + \
    ".(L1-L2-L3) - " + \
    df_inv["Dist"].round(1).astype(str) + "m"

df_inv['Script 1'] = "-insert Inversor " + \
                     df_inv['X'].astype(str) + "," +\
                     df_inv['Y'].astype(str) + \
                     " 1 1 0"

df_inv['Script 2'] = "(command-s \"._mtext\" \""+ df_inv['X tag'].astype(str) + "," + \
                     df_inv['Y tag'].astype(str) + \
                     "\" \"h\" 30  \"j\" \"ML\" \"r\" -90  \"w\" 0 \"" + \
                     df_inv['TAG'].astype(str) + \
                     "\" \"\")"

df_inv['Script 3'] = "(command-s \"._mtext\" \""+ df_inv['X cabo'].astype(str) + "," + \
                     df_inv['Y cabo'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"TL\" \"r\" 0  \"w\" 0 \"" + \
                     df_inv['TAG Cabo CA'].astype(str) + \
                     "\" \"\")"


with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - Script Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_unifilar_inversor\n\n')
    f.write('\n'.join(df_inv["Script 1"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - Script TAG Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_TAG_inversor\n\n')
    f.write('\n'.join(df_inv["Script 2"].astype(str)) + '\n')

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - Script Cabos CA.scr", 'w') as f:
    f.write('-LAYER\nSET\nCA_Cabos\n\n')
    f.write('\n'.join(df_inv["Script 3"].astype(str)) + '\n')

String distances
=
Reads the string tag (column H) and distance (column J) of the Riodades `STR` sheet as `TAG` / `Distance`. The distance is used to order the strings inside each inverter.

This is the same data already loaded in `df_str` (`Tag completa` and `Dist`), so the extra read is redundant.

In [2]:
String_Distance = pd.read_excel(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\309 - Zagope - UVF Riodades\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Riodades - Unifilar BT.xlsx", sheet_name = "STR", names = ["TAG", "Distance"], usecols = "H,J")

print(String_Distance)

                TAG  Distance
0     STR-1.1.01.01    156.40
1     STR-1.1.01.02    156.48
2     STR-1.1.01.03     77.02
3     STR-1.1.01.04     77.10
4     STR-1.1.01.05     37.42
...             ...       ...
2853  STR-4.2.20.14      7.92
2854  STR-4.2.20.15     87.22
2855  STR-4.2.20.16     87.30
2856  STR-4.2.20.17    166.60
2857  STR-4.2.20.18    166.68

[2858 rows x 2 columns]


Input check
=
Prints the string table (`df_str`) and the inverter table with the computed positions and scripts (`df_inv`).

In [3]:
print(df_str)
print(df_inv)

            Tag Type  Type N  UFV  PST  INV  STR   Tag completa  POT    Dist  \
0     1.1.01.01    T       1    1    1    1    1  STR-1.1.01.01  710  156.40   
1     1.1.01.02    B       2    1    1    1    2  STR-1.1.01.02  710  156.48   
2     1.1.01.03    T       1    1    1    1    3  STR-1.1.01.03  710   77.02   
3     1.1.01.04    B       2    1    1    1    4  STR-1.1.01.04  710   77.10   
4     1.1.01.05    C       3    1    1    1    5  STR-1.1.01.05  710   37.42   
...         ...  ...     ...  ...  ...  ...  ...            ...  ...     ...   
2853  4.2.20.14    B       2    4    2   20   14  STR-4.2.20.14  710    7.92   
2854  4.2.20.15    T       1    4    2   20   15  STR-4.2.20.15  710   87.22   
2855  4.2.20.16    B       2    4    2   20   16  STR-4.2.20.16  710   87.30   
2856  4.2.20.17    T       1    4    2   20   17  STR-4.2.20.17  710  166.60   
2857  4.2.20.18    B       2    4    2   20   18  STR-4.2.20.18  710  166.68   

         GRP  Qty. of Strings  PD  
0  

Building the string table
=
- Renames the first `df_inv` column (the inverter tag) to `TAG INV` for the merge with the strings later.
- `df_strings` takes, per string, the tag, `UFV`, `PST`, `INV`, `STR`, `Type`, `Type N`, `Qty. of Strings`, `PD` and `Module Power` (`POT`) from the `STR` sheet. `PV`, `X`, `Y`, `XX`, `YY` and the script columns start empty.
- Left-merge with `String_Distance` on `TAG` adds `Distance`. A string tag repeated in the `STR` sheet would duplicate rows.

In [4]:
df_inv = df_inv.rename(columns={df_inv.columns[0]: "TAG INV"})

 # Strings TAGs
df_strings = pd.DataFrame({
    'TAG': df_str["Tag completa"],
    'TAG Especial': None,
    'TAG INV': None,
    'TAG PST': None,
    'UFV': df_str["UFV"],
    'PST': df_str["PST"],
    'INV': df_str["INV"],
    'STR': df_str["STR"],
    'Type': df_str["Type"],
    'Type N': df_str["Type N"],
    'Qty. of Strings': df_str["Qty. of Strings"],
    'PD': df_str["PD"],
    'PV': None,
    'X': None,
    'Y': None,
    'XX': None,
    'YY': None,
    'Script TAG String': None,
    'Script Potência': None,
    'Module Power': df_str["POT"],
})

df_strings = df_strings.merge(String_Distance, on="TAG", how="left")
print(df_strings)

                TAG TAG Especial TAG INV TAG PST  UFV  PST  INV  STR Type  \
0     STR-1.1.01.01         None    None    None    1    1    1    1    T   
1     STR-1.1.01.02         None    None    None    1    1    1    2    B   
2     STR-1.1.01.03         None    None    None    1    1    1    3    T   
3     STR-1.1.01.04         None    None    None    1    1    1    4    B   
4     STR-1.1.01.05         None    None    None    1    1    1    5    C   
...             ...          ...     ...     ...  ...  ...  ...  ...  ...   
2853  STR-4.2.20.14         None    None    None    4    2   20   14    B   
2854  STR-4.2.20.15         None    None    None    4    2   20   15    T   
2855  STR-4.2.20.16         None    None    None    4    2   20   16    B   
2856  STR-4.2.20.17         None    None    None    4    2   20   17    T   
2857  STR-4.2.20.18         None    None    None    4    2   20   18    B   

      Type N  ...  PD    PV     X     Y    XX    YY Script TAG String  \
0 

String types and ordering inside each inverter
=
- `TAG Especial` = `(Type) STR-...` (e.g. `(T) STR-1.1.01.01`). `TAG INV` / `TAG PST` are cut from the string tag.
- T-C-B exception (from the code comment): in an inverter with **17 strings and exactly one C string**, the B strings get `Type N = 5` and the C string `Type N = 4`. Sorting by `Type N` then gives T-C-B (1-4-5) instead of the usual T-B-C (1-2-3), so the C string shares an MPPT with T strings.
  - The loop looks each row up by `TAG` and only updates the first match, so a duplicated string tag is only fixed once.
- Ordering by distance: `sorting_distance` is `−Distance` for B strings (`Type N` 2 and 5) and `Distance` for the others, so inside each type T / C strings go nearest first and B strings farthest first.
- Sorts by `UFV, PST, INV, Type N, sorting_distance` and drops the helper column. `ascending_types` is defined but not used (only `descending_types` is).

In [5]:

df_strings["TAG Especial"] = "("+df_strings['Type'].astype(str)+") "+df_strings['TAG'].astype(str)
df_strings["TAG INV"] = "INV-" + df_strings["TAG"].astype(str).str.slice(4, 10)
df_strings["TAG PST"] = "PST-" + df_strings["TAG"].astype(str).str.slice(4, 7)


all_tags = df_strings["TAG INV"].unique()
c_counts = (df_strings[df_strings["Type N"] == 3]["TAG INV"].value_counts().reindex(all_tags, fill_value=0)) # Counts how many C-type Strings there are in all inverters

# This gets the exception where the C-string should share the MPPT with the T-string so that less MPPTs have different types of strings. So it changes the Type N for B-strings to 5 and the Type N for C-strings to 4 to order them T-C-B (1-4-5) instead of the usual T-B-C (1-2-3).
for tag in df_strings["TAG"]:
    index = df_strings[df_strings["TAG"] == tag].index[0]
    tag_inv = df_strings.at[index, "TAG INV"]
    c_count = c_counts[tag_inv]
    if df_strings.at[index, "Qty. of Strings"] == 17 and c_count == 1:
        if df_strings.at[index, "Type N"] == 2:
            df_strings.at[index, "Type N"] = 5
        elif df_strings.at[index, "Type N"] == 3:
            df_strings.at[index, "Type N"] = 4

# Define which Type N values should have ascending vs descending Distance
ascending_types = [1,3,4]
descending_types = [2,5]


# Create modified distance for sorting
df_strings['sorting_distance'] = np.where(
    df_strings['Type N'].isin(descending_types),
    -df_strings['Distance'],
    df_strings['Distance']
)

df_strings = df_strings.sort_values(
    by=["UFV", "PST", "INV", "Type N", "sorting_distance"]
).reset_index(drop=True)


df_strings = df_strings.drop("sorting_distance", axis=1)

Slots, string scripts and output files
=
- `Position in group` = running count **per `PD`**, and `PV = PATTERNS[PD][position mod len(PATTERNS[PD])]` is the slot (row) of the string next to its inverter.
  - Warning: the counter runs per pattern over the whole plant, not per inverter. It only lines up if every inverter has exactly as many strings as its pattern has slots (18 / 17 / 16). One inverter with a different count shifts the slots of every following inverter with that pattern.
- Left-merge with the inverter `X`, `Y` on `TAG INV`. `df_strings` already has empty `X` / `Y` columns, so pandas names them `X_x` / `Y_x` (empty) and `X_y` / `Y_y` (inverter position).
  - `X_x` / `Y_x` are then overwritten with the **string tag** position: `X_inv + row 9`, `Y_inv + row 10 + row 11 · (PV − 1)`.
  - `XX` / `YY` hold the **module power** position: `X_inv + row 12`, `Y_inv + row 13 + row 14 · (PV − 1)`.
- `Script TAG String`: MText of `TAG Especial`, height 20, justification MR. `Script Potência`: MText of the module power, height 20, justification MC.
- Module power comes from the `POT` column (`Module Power`).
- Outputs in `Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar`: `Riodades - Diagrama Unifilar BT.xlsx` (the full string table), `Riodades - tag string.scr` (layer `TAG_strings`) and `Riodades - potencia.scr` (layer `Potência_Módulo`).
- Prints the time elapsed since the first cell.

In [6]:

df_strings["Position in group"] = df_strings.groupby("PD").cumcount()

# adicionar C,T,B
df_strings['PV'] = df_strings.apply(
    lambda row: PATTERNS[row['PD']][row['Position in group'] % len(PATTERNS[row['PD']])],
    axis=1
)

df_strings = df_strings.merge(df_inv[["TAG INV", "X", "Y"]], on="TAG INV", how="left")

df_strings['X_x'] = df_strings['X_y'].astype(float)+df_parameter.iloc[9,1]
df_strings['Y_x'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[10,1]+df_parameter.iloc[11, 1]*(df_strings["PV"].astype(float)-1)
df_strings['XX'] = df_strings['X_y'].astype(float)+df_parameter.iloc[12,1]
df_strings['YY'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[13,1]+df_parameter.iloc[14, 1]*(df_strings["PV"].astype(float)-1)

df_strings['Script TAG String'] = "(command-s \"._mtext\" \""+ df_strings['X_x'].astype(str) + "," + \
                     df_strings['Y_x'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MR\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['TAG Especial'].astype(str) + \
                     "\" \"\")"

df_strings['Script Potência'] = "(command-s \"._mtext\" \""+ df_strings['XX'].astype(str) + "," + \
                     df_strings['YY'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['Module Power'].astype(str) + \
                     "\" \"\")"

df_strings.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - Diagrama Unifilar BT.xlsx")

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - tag string.scr", 'w') as f:
    f.write('-LAYER\nSET\nTAG_strings\n\n')
    f.write('\n'.join(df_strings["Script TAG String"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Diagrama Unifilar\Riodades - potencia.scr", 'w') as f:
    f.write('-LAYER\nSET\nPotência_Módulo\n\n')
    f.write('\n'.join(df_strings["Script Potência"].astype(str)) + '\n')


end = time.time()
elapsed = end - start
print(elapsed)

4.154940366744995
